# ANNNI phase diagram from PennyLane circuits

Loschmidt echo readout, fourth-order Suzuki evolution, depolarizing noise charged per
two-qubit gate, finite shots. No ancilla and no controlled gates anywhere.

**What is quantum, what is classical.** The circuit returns one number per time point, the
return probability $P(t)=|\langle\psi_0|U(t)|\psi_0\rangle|^2$. Everything downstream —
ESPRIT spectral estimation — is classical, exactly as on hardware. Exact diagonalisation
appears only as a labelled benchmark.

**Why the echo rather than a Hadamard test.** The Hadamard test gives the complex amplitude
$g(t)$, but needs an ancilla controlling every gate: 21 entangling gates per second-order
step against 9 for the uncontrolled circuit, and an ancilla that must reach all six qubits.
The echo needs neither. It gives only $|g|^2$, so absolute energies are lost, but its
spectrum consists of the level *differences* — and the gap is the only thing the phase
detector uses. The cost is a denser spectrum, $\sim N^2$ lines instead of $N$, which forces
finer time sampling to avoid aliasing.

**Runtime** 20–30 min at the defaults. The heavy cells print progress and `PARAMS` in cell 3
controls the cost. Drop the smallest $\Gamma$ to roughly halve it.


In [ ]:
import time
import numpy as np
import pennylane as qml
import matplotlib.pyplot as plt
from scipy.linalg import expm
from scipy.sparse.linalg import eigsh

print("pennylane", qml.__version__, "| numpy", np.__version__)


## 1. Parameters

`GAMMA_TARGETS` rather than raw error rates: the decoherence rate $\Gamma$ is what enters
the physics, and cell 8 calibrates $\Gamma(p)$ so these can be inverted into per-gate error
rates. That keeps the experiment comparable across different circuit choices.

`ORDER` switches the product formula. Both are provided because the choice is not obvious:
fourth order is far more accurate per step but uses five times the gates, and with noise
charged per gate that costs a factor of two in $\Gamma$. See cell 8.


In [ ]:
N     = 6         # register qubits, no ancilla
KAPPA = 0.2       # frustration, fixed for the h scans
ORDER = 4         # product formula order: 2 or 4
DT    = 0.30      # Trotter step (use 0.15 with ORDER = 2)
SHOTS = 10_000    # shots per return-probability estimate
SEED  = 20260921

ORDER_CLEAN = 16  # ESPRIT model order without noise
ORDER_NOISY = 24  # and with it. These differ because they must: a subspace
                  # method can afford as many modes as the SNR supports and no
                  # more, so the order is chosen on data matched to each case.

GAMMA_TARGETS = [0.02, 0.05, 0.10]
H_LIST = [0.4, 0.5, 0.6, 0.7, 0.8, 1.0, 1.2, 1.5]

GRID_K, GRID_H, GRID_NT = np.linspace(0.0, 1.0, 21), np.linspace(0.1, 1.7, 21), 400

FLOOR = 1 / 2**N     # a fully depolarised state still overlaps |psi_0> by 2^-N
rng = np.random.default_rng(SEED)


## 2. Model

$$H=-J\sum_i Z_iZ_{i+1}+J\kappa\sum_i Z_iZ_{i+2}-Jh\sum_i X_i,\qquad J=1$$


In [ ]:
def hamiltonian(n, kappa, h):
    coeffs, terms = [], []
    for i in range(n - 1): coeffs.append(-1.0);   terms.append(qml.Z(i) @ qml.Z(i + 1))
    for i in range(n - 2): coeffs.append(kappa);  terms.append(qml.Z(i) @ qml.Z(i + 2))
    for i in range(n):     coeffs.append(-h);     terms.append(qml.X(i))
    return qml.Hamiltonian(coeffs, terms)


def exact_levels(n, kappa, h, k=4):
    "BENCHMARK ONLY."
    m = hamiltonian(n, kappa, h).sparse_matrix(wire_order=range(n)).tocsc().real
    return np.sort(eigsh(m, k=k, which="SA", return_eigenvectors=False))


print("levels at (0.2, 0.7):", np.round(exact_levels(N, 0.2, 0.7), 4))


## 3. Reference state

$H$ commutes with the spin flip $P=\prod_i X_i$, and $|+\rangle^{\otimes N}$ is an
eigenstate of it — so its overlap with the odd-parity partner of the ferromagnetic doublet
vanishes identically, and the splitting used as the phase detector disappears from the
signal at any resolution. The cell below measures that. The reference used instead is an
equal superposition of $|+\rangle^{\otimes N}$, the ferromagnet and the period-four state.


In [ ]:
def reference_state(n):
    # Five components: |+>^n, the ferromagnet, the period-four state, and two
    # domain walls. The domain walls are not decoration -- see the cell below.
    v = np.ones(2**n, dtype=complex) / 2 ** (n / 2)
    for pat in ("0", "0011", "000111", "001111"):
        v[int((pat * n)[:n], 2)] += 1.0
    return v / np.linalg.norm(v)


def reference_3(n):
    "The earlier three-component reference, kept for the comparison below."
    v = np.ones(2**n, dtype=complex) / 2 ** (n / 2)
    for pat in ("0", "0011"):
        v[int((pat * n)[:n], 2)] += 1.0
    return v / np.linalg.norm(v)


def _product(n, theta):
    one = np.array([np.cos(theta / 2), np.sin(theta / 2)], dtype=complex)
    v = np.ones(1, dtype=complex)
    for _ in range(n): v = np.kron(v, one)
    return v


REFL = np.array([int(format(x, f"0{N}b")[::-1], 2) for x in range(2**N)])

print(f"{'reference':>16} {'|<r|R|r>|':>10}   parity under reflection")
for name, psi in (("|+>^N", _product(N, np.pi / 2)),
                  ("3-component", reference_3(N)),
                  ("5-component", reference_state(N))):
    print(f"{name:>16} {abs(psi.conj() @ psi[REFL]):10.4f}")
print("\n1.000 means exactly reflection-even, hence blind to every odd eigenstate.\n")

print(f"{'(kappa,h)':>12} {'R(E0)':>6} {'R(E1)':>6} | "
      f"{'|+>^N':>9} {'3-comp':>9} {'5-comp':>9}   <- |<r|E1>|^2")
for k, h in ((0.2, 0.5), (0.6, 1.2), (0.4, 1.0)):
    m = hamiltonian(N, k, h).sparse_matrix(wire_order=range(N)).tocsc().real
    w, vecs = eigsh(m, k=2, which="SA"); vecs = vecs[:, np.argsort(w)]
    par = [float(vecs[:, j] @ vecs[REFL, j]) for j in range(2)]
    row = f"{str((k,h)):>12} {par[0]:+6.1f} {par[1]:+6.1f} | "
    for psi in (_product(N, np.pi / 2), reference_3(N), reference_state(N)):
        row += f"{abs(psi.conj()@vecs[:,1])**2:9.2e} "
    print(row)
print("\nWhere E1 is reflection-odd the first two references see nothing at all.")
print("A domain wall is maximally asymmetric under R, which is what fixes it.")


## 4. Circuit

A second-order Strang step realises $e^{-icP\delta t}$ as `IsingZZ(2c dt)` for $P=Z_iZ_j$
and `RX(2c dt)` for $P=X_i$. Fourth order is the Suzuki fractal, five second-order
substeps with $s=1/(4-4^{1/3})$.

Depolarizing noise follows **every two-qubit gate**, which is what makes the comparison
between the two orders honest: the fourth-order step is five times longer and pays for it.


In [ ]:
SUZUKI = 1.0 / (4.0 - 4.0 ** (1 / 3))


def step2(n, kappa, h, dt, p=0.0):
    def zz(a, b, ang):
        qml.IsingZZ(ang, wires=[a, b])
        if p > 0:
            qml.DepolarizingChannel(p, wires=a); qml.DepolarizingChannel(p, wires=b)
    for i in range(n): qml.RX(-h * dt, wires=i)
    for i in range(n - 1): zz(i, i + 1, -2.0 * dt)
    for i in range(n - 2): zz(i, i + 2, 2.0 * kappa * dt)
    for i in range(n): qml.RX(-h * dt, wires=i)


def step4(n, kappa, h, dt, p=0.0):
    for f in (SUZUKI, SUZUKI, 1 - 4 * SUZUKI, SUZUKI, SUZUKI):
        step2(n, kappa, h, f * dt, p)


STEP = step4 if ORDER == 4 else step2


def two_qubit_count(step, dt):
    with qml.queuing.AnnotatedQueue() as q:
        step(N, KAPPA, 0.7, dt)
    ops = qml.tape.QuantumScript.from_queue(q).operations
    return sum(1 for o in ops if len(o.wires) == 2)


print(f"two-qubit gates per step:  order 2 = {two_qubit_count(step2, DT)}"
      f"   order 4 = {two_qubit_count(step4, DT)}")
print(f"using order {ORDER} at dt = {DT}:"
      f" {two_qubit_count(STEP, DT)/DT:.0f} two-qubit gates per unit evolution time")
print()
dev_draw = qml.device("default.mixed", wires=N)
print(qml.draw(qml.QNode(lambda: (step2(N, KAPPA, 0.7, DT, 0.001),
                                  qml.expval(qml.Z(0)))[1], dev_draw),
               max_length=100)())


## 5. The echo signal

On hardware: prepare $|\psi_0\rangle=V|0\rangle$, evolve, apply $V^\dagger$, count the
all-zeros outcome. That probability is $|\langle\psi_0|U(t)|\psi_0\rangle|^2$, which is the
expectation of the projector onto $|\psi_0\rangle$ — so `qml.Projector` measures exactly the
quantity the hardware protocol returns, and `qml.Snapshot` reads it after every step in a
single run, $O(n_t)$ instead of $O(n_t^2)$.

Shots are binomial on that probability, which is exactly the measurement statistics.
Keeping sampling separate from the circuit means the circuit runs once per setting while
the readout can be resampled for free.


In [ ]:
def echo_exact(n, kappa, h, n_times, dt, p=0.0, step=None):
    "Return probability after each step, before readout sampling."
    step = step or STEP
    psi0 = reference_state(n)
    dev = qml.device("default.mixed" if p > 0 else "default.qubit", wires=n)

    def circ():
        qml.StatePrep(psi0, wires=range(n))
        obs = qml.Projector(psi0, wires=range(n))
        qml.Snapshot(measurement=qml.expval(obs))
        for _ in range(n_times - 1):
            step(n, kappa, h, dt, p)
            qml.Snapshot(measurement=qml.expval(obs))
        return qml.expval(obs)

    snaps = qml.snapshots(qml.QNode(circ, dev))()
    return np.array([float(v) for k, v in snaps.items() if k != "execution_results"])


def apply_shots(prob, shots, rng):
    return rng.binomial(shots, np.clip(prob, 0, 1)) / shots


t0 = time.time(); P = echo_exact(N, KAPPA, 0.7, 30, DT)
print(f"30 points in {time.time()-t0:.1f}s;  P(0) = {P[0]:.4f}  (expect 1)")


## 6. Check A — the circuit reproduces exact dynamics

Only Trotter error should remain. This also shows the gap the two orders open up.


In [ ]:
def echo_reference(h, nt, dt):
    "BENCHMARK ONLY."
    hm = hamiltonian(N, KAPPA, h).sparse_matrix(wire_order=range(N)).toarray()
    psi = reference_state(N)
    u = expm(-1j * dt * hm)
    cur, out = psi.copy(), []
    for _ in range(nt):
        out.append(abs(psi.conj() @ cur) ** 2)
        cur = u @ cur
    return np.array(out)


print(f"{'dt':>6} {'order 2':>11} {'order 4':>11}")
for dt in (0.15, 0.30, 0.50):
    ref = echo_reference(0.7, 30, dt)
    e2 = np.abs(echo_exact(N, KAPPA, 0.7, 30, dt, step=step2) - ref).max()
    e4 = np.abs(echo_exact(N, KAPPA, 0.7, 30, dt, step=step4) - ref).max()
    print(f"{dt:6.2f} {e2:11.3e} {e4:11.3e}")


## 7. Check B — aliasing

The echo carries every level *difference*, so its bandwidth is the full spectral width $W$,
not the spectral radius. Strict Nyquist would demand $\delta t<\pi/W$, but the reference
state only populates low-lying levels, so the high-frequency differences carry no weight
and the practical limit is looser. This finds where it actually breaks.


In [ ]:
hs = hamiltonian(N, KAPPA, 1.5).sparse_matrix(wire_order=range(N)).tocsc().real
lo = eigsh(hs, k=1, which="SA", return_eigenvectors=False)[0]
hi = eigsh(hs, k=1, which="LA", return_eigenvectors=False)[0]
print(f"h=1.5 spectrum [{lo:.2f}, {hi:.2f}], width {hi-lo:.2f}")
print(f"strict Nyquist would need dt < pi/W = {np.pi/(hi-lo):.3f}\n")


def esprit(x, dt, order):
    m = len(x) // 2
    h0 = np.array([x[i:i + m] for i in range(len(x) - m)])
    u, _, _ = np.linalg.svd(h0, full_matrices=False)
    uu = u[:, :order]
    return np.sort(np.angle(np.linalg.eigvals(np.linalg.pinv(uu[:-1]) @ uu[1:])) / -dt)


def gap_from_echo(prob, dt, order):
    # Smallest positive difference frequency; DC excluded. `order` is required
    # rather than defaulted: the right value depends on the signal-to-noise
    # ratio of the record it is applied to, and using a noiseless choice on
    # noisy data was the most expensive mistake in this project.
    f = esprit(prob.astype(complex), dt, order)
    pos = np.sort(f[f > 5e-3])
    return float(pos[0]) if len(pos) else np.nan


ex15 = np.diff(exact_levels(N, KAPPA, 1.5)[:2])[0]
print(f"gap at h=1.5, exact {ex15:.4f}")
for dt in (0.15, 0.30, 0.40, 0.60):
    v = gap_from_echo(echo_exact(N, KAPPA, 1.5, int(90 / dt), dt), dt, ORDER_CLEAN)
    print(f"  dt={dt:.2f}  {v:8.4f}  {'ok' if abs(v-ex15) < 0.15 else 'ALIASED'}")


## 8. Check C — calibrating $\Gamma(p)$

The echo decays toward $2^{-N}$, not zero: a fully depolarised state still overlaps
$|\psi_0\rangle$ by that much. The envelope is fitted on $|P-2^{-N}|$.

This also settles the order question. With noise per gate, the fourth-order step should
cost about $(45/0.30)/(9/0.15)=2.5\times$ the second-order one in $\Gamma$.


In [ ]:
def calibrate(step, dt, ps=(0.0005, 0.001, 0.002), nt=50):
    t = dt * np.arange(nt)
    amp_c = np.abs(echo_exact(N, KAPPA, 0.7, nt, dt, step=step) - FLOOR)
    ok = amp_c > 0.05
    rates = []
    for p in ps:
        a = np.abs(echo_exact(N, KAPPA, 0.7, nt, dt, p, step=step) - FLOOR)
        sl, _ = np.polyfit(t[ok], np.log(a[ok] / amp_c[ok]), 1)
        rates.append(-sl)
    return float(np.polyfit(ps, rates, 1)[0])


g2 = calibrate(step2, 0.15)
g4 = calibrate(step4, 0.30)
print(f"order 2, dt=0.15:  Gamma = {g2:6.1f} p")
print(f"order 4, dt=0.30:  Gamma = {g4:6.1f} p     ratio {g4/g2:.2f}x")
print(f"gate-count prediction: {(45/0.30)/(9/0.15):.2f}x")
print("\nFourth order buys accuracy and pays for it in decoherence. On a device with a")
print("fixed per-gate error rate, second order at dt=0.15 reaches a smaller Gamma.")
GAMMA_PER_P = g4 if ORDER == 4 else g2
print(f"\nusing Gamma = {GAMMA_PER_P:.1f} p")


## 9. The resolution experiment

Depolarizing noise damps the signal; finite shots put a floor under it. The record is
usable only while the envelope stays above that floor, which caps the evolution time and
so the spectral resolution:

$$T_{\max}=\frac{\ln\sqrt{M}}{\Gamma},\qquad \delta E\simeq\frac{2\pi}{T_{\max}}$$

The prediction is that the gap is recoverable where $\Delta(h)>\delta E$ and not below.

**Expensive cell**, 15–25 min. Progress after each rate.


In [ ]:
gaps_exact = np.array([np.diff(exact_levels(N, KAPPA, h)[:2])[0] for h in H_LIST])
results, meta = {}, {}

for gamma in GAMMA_TARGETS:
    p = gamma / GAMMA_PER_P
    T = np.log(np.sqrt(SHOTS)) / gamma
    nt = max(32, int(T / DT))
    meta[gamma] = (p, nt, 2 * np.pi / T)
    t0, est = time.time(), []
    for h in H_LIST:
        base = echo_exact(N, KAPPA, h, nt, DT, p)
        est.append(np.median([gap_from_echo(apply_shots(base, SHOTS,
                                                        np.random.default_rng(SEED + s)), DT, ORDER_NOISY)
                              for s in range(3)]))
    results[gamma] = np.array(est)
    print(f"Gamma={gamma:.3f}  p={p:.2e}  nt={nt:5d}  dE={meta[gamma][2]:.3f}"
          f"   [{time.time()-t0:.0f}s]", flush=True)


In [ ]:
fig, (ax, sx) = plt.subplots(2, 1, figsize=(5.6, 4.6), sharex=True,
                             gridspec_kw={"height_ratios": [3, 1.2], "hspace": 0.08})
colors = plt.cm.viridis(np.linspace(0.15, 0.8, len(GAMMA_TARGETS)))

ax.semilogy(H_LIST, gaps_exact, "k-", lw=1.5, label=r"exact $E_1-E_0$")
for c, gamma in zip(colors, GAMMA_TARGETS):
    dE = meta[gamma][2]
    ax.axhline(dE, color=c, ls="--", lw=1.0)
    ax.text(H_LIST[-1], dE * 1.12, rf"$\delta E(\Gamma={gamma})$", color=c,
            fontsize=7, ha="right")
    ax.axvline(float(np.interp(dE, gaps_exact, H_LIST)), color=c, ls=":", lw=0.8)
ax.set_ylabel("level spacing"); ax.legend(fontsize=8); ax.tick_params(labelbottom=False)

for row, (c, gamma) in enumerate(zip(colors, GAMMA_TARGETS)):
    good = np.abs(results[gamma] - gaps_exact) < 0.15 * gaps_exact + 0.02
    y = np.full(len(H_LIST), len(GAMMA_TARGETS) - 1 - row, float)
    sx.plot(np.array(H_LIST)[good], y[good], "o", color=c, ms=5)
    sx.plot(np.array(H_LIST)[~good], y[~good], "o", mfc="white", mec=c, ms=5, mew=1.1)
    sx.axvline(float(np.interp(meta[gamma][2], gaps_exact, H_LIST)), color=c, ls=":", lw=0.8)
sx.set_yticks(range(len(GAMMA_TARGETS)))
sx.set_yticklabels([rf"$\Gamma$={g}" for g in reversed(GAMMA_TARGETS)], fontsize=7)
sx.set_ylim(-0.7, len(GAMMA_TARGETS) - 0.3); sx.set_xlabel("$h$")
sx.tick_params(axis="y", length=0)
plt.savefig("manuscript/resolvability_echo.pdf", bbox_inches="tight"); plt.show()

print(f"{'Gamma':>7} {'dE':>8} {'predicted h*':>13} {'observed onset':>15}")
for gamma in GAMMA_TARGETS:
    good = np.abs(results[gamma] - gaps_exact) < 0.15 * gaps_exact + 0.02
    onset = float(np.array(H_LIST)[good].min()) if good.any() else float("nan")
    print(f"{gamma:7.3f} {meta[gamma][2]:8.3f}"
          f" {float(np.interp(meta[gamma][2], gaps_exact, H_LIST)):13.3f} {onset:15.3f}")
print("\nThe onset tracks the prediction, but the estimator needs the gap to clear dE by")
print("roughly a factor of two rather than merely exceed it -- marginal cases still fail.")


## 10. Phase map from the circuit

A $21\times21$ grid, noiseless, with the gap read from the echo at each site.

The lower part of the map comes out blank, and that is not a grid artefact. Even with no
noise at all, a record of length $T=n_t\,\delta t$ resolves nothing finer than
$\delta E\simeq2\pi/T$, and the doublet splitting falls below $10^{-3}$ deep in the ordered
phase. The same resolution limit that decoherence imposes in cell 9 is imposed here by the
finite record. The dashed contour marks where the exact gap equals $\delta E$: the map goes
blind exactly there, which is the prediction rather than a failure.

About 6 minutes. Raising `GRID_NT` pushes the blind region down, at linear cost.


In [ ]:
gap_map = np.zeros((len(GRID_H), len(GRID_K)))
exact_map = np.zeros_like(gap_map)
t0 = time.time()
for a, h in enumerate(GRID_H):
    for b, k in enumerate(GRID_K):
        gap_map[a, b] = gap_from_echo(echo_exact(N, k, h, GRID_NT, DT), DT, ORDER_CLEAN)
        exact_map[a, b] = np.diff(exact_levels(N, k, h, 3)[:2])[0]
    print(f"h={h:.2f}  [{time.time()-t0:.0f}s]", flush=True)

DE_RECORD = 2 * np.pi / (GRID_NT * DT)      # resolution set by the record length alone
resolved = np.abs(gap_map - exact_map) < 0.15 * exact_map + 1e-3
print(f"\nrecord T = {GRID_NT*DT:.0f}, so dE = {DE_RECORD:.3f}")
print(f"sites recovered: {resolved.sum()}/{resolved.size}")
print(f"  where exact gap > dE : {resolved[exact_map > DE_RECORD].mean():.0%}")
print(f"  where exact gap < dE : {resolved[exact_map < DE_RECORD].mean():.0%}")


def ferro_para(k):
    k = np.clip(k, 1e-6, 0.5)
    return (1 - k) / k * (1 - np.sqrt((1 - 3 * k + 4 * k**2) / (1 - k)))


fig, axes = plt.subplots(1, 2, figsize=(9.5, 3.5), constrained_layout=True)
kk = np.linspace(0.001, 0.499, 200); ka = np.linspace(0.5, 1, 100)
for ax, d, title in ((axes[0], gap_map, "gap from the echo circuit"),
                     (axes[1], exact_map, "exact $E_1-E_0$ (benchmark)")):
    im = ax.pcolormesh(GRID_K, GRID_H, np.clip(d, 1e-3, None), cmap="magma",
                       norm=plt.matplotlib.colors.LogNorm(1e-3, 3), shading="nearest")
    ax.plot(kk, ferro_para(kk), "w--", lw=1.1)
    ax.plot(ka, 1.05 * np.sqrt(ka - 0.5), "w:", lw=1.1)
    ax.contour(GRID_K, GRID_H, exact_map, levels=[DE_RECORD],
               colors="cyan", linewidths=1.4, linestyles="--")
    ax.set_xlabel(r"$\kappa$"); ax.set_ylabel("$h$"); ax.set_title(title, fontsize=10)
    fig.colorbar(im, ax=ax)
axes[0].plot([], [], "c--", lw=1.4, label=rf"exact gap $=\delta E={DE_RECORD:.2f}$")
axes[0].plot([], [], "w--", lw=1.1, label="analytic boundaries")
axes[0].legend(fontsize=7, loc="upper left")
plt.savefig("manuscript/phase_map_echo.pdf", bbox_inches="tight"); plt.show()


## 11. What this establishes

1. The echo circuit reproduces exact dynamics up to Trotter error, with no ancilla and no
   controlled gates (cell 6).
2. Its spectrum carries level differences, so it aliases at a coarser $\delta t$ than the
   amplitude signal would; the practical limit is looser than strict Nyquist because the
   reference state populates only low-lying levels (cell 7).
3. With noise charged per two-qubit gate, fourth order costs about twice the $\Gamma$ of
   second order at equal accuracy target. Its extra accuracy does not pay for itself here,
   because Trotter error was never the binding constraint — $\Gamma$ was (cell 8).
4. The gap is recoverable where $\Delta(h)>\delta E(\Gamma)$ and lost below it. Noise hides
   the boundary rather than moving it (cell 9).
5. The same limit appears with no noise at all. In the phase map of cell 10 the record
   length alone fixes $\delta E=2\pi/T$, and the map is recovered above the contour where
   the gap crosses it and blank below. Decoherence and a short record are two ways of
   buying the same resolution.

**Limits.** `N = 6` and a coarse grid are what a density-matrix simulator affords. The
estimator needs roughly a factor of two of margin over $\delta E$, not a bare crossing, so
the criterion predicts the onset only to within that. The scaling law
$|h^*-h_c|\sim\Gamma^{1/z\nu}$ needs more usable $\Gamma$ values and several $\kappa$, and
is **not** tested here; widening the usable range in $\Gamma$ calls for denoising the
signal subspace, not for changing the circuit.
